In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import ast
import cv2

# -----------------------------
# CONFIG: change if needed
# -----------------------------
X_COL = "ear_mid_x"
Y_COL = "ear_mid_y"
FRAME_COL = "beh_frame_idx"     # frame number column in aligned CSV
USE_FRAME_COL = True            # True = slice by FRAME_COL range (recommended)

VIDEO_COL = "beh_vid_path"           # column in aligned CSV that contains behavior video path

ARENA_ALPHA = 0.6               # background visibility (0..1)
INVERT_Y = True                 # True for video pixel coords

LABELS = [
    "left_small_loop",
    "left_big_loop",
    "right_small_loop",
    "right_big_loop",
    "non_characteristic",
]

# -----------------------------
# LOAD CSVs
# -----------------------------
TRIALS_CSV = r"C:\Users\psych-aalab\Desktop\zenon_frametest\sklearn_testing\trials_by_session.csv"
ALIGNED_CSV = r"C:\Users\psych-aalab\Desktop\zenon_frametest\sklearn_testing\aligned_sessions_with_rois.csv"

# trials_by_session (WIDE format: columns are sessions like '11/6/25', cells like '(2654, 2947)')
trials_df = pd.read_csv(TRIALS_CSV)

# aligned_sessions_with_rois (LONG format: has 'session_id' column)
aligned_df = pd.read_csv(ALIGNED_CSV)

# -----------------------------
# CONVERT trials_df -> trials_by_session dict
# -----------------------------
def parse_pair_cell(v):
    """Parse '(2654, 2947)' or '[2654, 2947]' or '2654,2947' into (2654, 2947)."""
    if pd.isna(v):
        return None

    s = str(v).strip()

    # Try safe literal eval: "(a,b)" or "[a,b]"
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (tuple, list)) and len(obj) == 2:
            return (int(obj[0]), int(obj[1]))
    except Exception:
        pass

    # Fallback: "a,b"
    parts = [p.strip() for p in s.replace("(", "").replace(")", "").split(",")]
    if len(parts) == 2:
        return (int(float(parts[0])), int(float(parts[1])))

    return None

trials_by_session = {}
for session_col in trials_df.columns:
    pairs = []
    for v in trials_df[session_col].dropna().tolist():
        pair = parse_pair_cell(v)
        if pair is not None:
            pairs.append(pair)
    trials_by_session[session_col] = pairs

# -----------------------------
# CONVERT aligned_df -> aligned_sessions_with_rois dict
# -----------------------------
required_cols = {"session_id", FRAME_COL, X_COL, Y_COL, VIDEO_COL}
missing = required_cols - set(aligned_df.columns)
if missing:
    raise ValueError(f"aligned_sessions_with_rois CSV missing columns: {missing}. Found: {list(aligned_df.columns)}")

aligned_sessions_with_rois = {
    sid: df.sort_values(FRAME_COL).reset_index(drop=True)
    for sid, df in aligned_df.groupby("session_id", sort=True)
}

# Keep only sessions present in both
common_sessions = sorted(set(trials_by_session.keys()) & set(aligned_sessions_with_rois.keys()))
if len(common_sessions) == 0:
    raise ValueError(
        "No matching session IDs between trials CSV (its column names) and aligned CSV ('session_id').\n"
        f"Trials sessions example: {list(trials_by_session.keys())[:5]}\n"
        f"Aligned sessions example: {list(aligned_sessions_with_rois.keys())[:5]}"
    )

trials_by_session = {k: trials_by_session[k] for k in common_sessions}
aligned_sessions_with_rois = {k: aligned_sessions_with_rois[k] for k in common_sessions}

# -----------------------------
# Arena frame cache + loader
# -----------------------------
arena_frame_cache = {}

def get_arena_frame(session):
    """Load + cache the FIRST frame of the behavior video for this session."""
    if session in arena_frame_cache:
        return arena_frame_cache[session]

    df = aligned_sessions_with_rois[session]
    video_path = df[VIDEO_COL].iloc[0]

    cap = cv2.VideoCapture(video_path)
    ret, frame = cap.read()
    cap.release()

    if not ret or frame is None:
        raise RuntimeError(f"Could not read first frame from video: {video_path}")

    # OpenCV returns BGR; matplotlib expects RGB
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    arena_frame_cache[session] = frame
    return frame

# -----------------------------
# Storage for labels
# -----------------------------
labels_records = []  # list of dicts -> DataFrame later

def get_trial_df(session, trial_idx):
    trials = trials_by_session[session]
    s, e = map(int, trials[trial_idx])

    df = aligned_sessions_with_rois[session]

    if USE_FRAME_COL:
        df_trial = df[(df[FRAME_COL] >= s) & (df[FRAME_COL] <= e)].copy()
    else:
        df_trial = df.iloc[s:e+1].copy()

    return df_trial, s, e

def plot_trial_trajectory(df_trial, session=None, s=None, e=None):
    x = df_trial[X_COL].to_numpy()
    y = df_trial[Y_COL].to_numpy()

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    fig, ax = plt.subplots(figsize=(6, 6))

    # ---- background arena image (first frame) ----
    arena_img = get_arena_frame(session)
    h, w = arena_img.shape[:2]

    ax.imshow(
        arena_img,
        extent=[0, w, h, 0],     # pixel coords (x: 0..w, y: 0..h) with y reversed
        interpolation="nearest",
        alpha=ARENA_ALPHA
    )

    # ---- trajectory (time-colored points helps a lot) ----
    if len(x) > 1:
        t = np.linspace(0, 1, len(x))
        ax.scatter(x, y, c=t, s=6)
        ax.plot(x, y, alpha=0.3)
    else:
        ax.plot(x, y)

    if len(x) > 0:
        ax.scatter([x[0]], [y[0]], marker="o", s=60, label="start")
        ax.scatter([x[-1]], [y[-1]], marker="x", s=60, label="end")

    ax.set_title(f"Session {session} | Frames {s}–{e} | N={len(x)}")
    ax.set_aspect("equal", adjustable="box")
    if INVERT_Y:
        ax.invert_yaxis()

    ax.legend(loc="best")
    ax.grid(True)
    plt.show()

# -----------------------------
# Widgets
# -----------------------------
session_ids = sorted(list(trials_by_session.keys()))

session_dropdown = widgets.Dropdown(
    options=session_ids,
    value=session_ids[0],
    description="Session:"
)

trial_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(trials_by_session[session_dropdown.value]) - 1,
    step=1,
    description="Trial:",
    continuous_update=False
)

label_dropdown = widgets.Dropdown(
    options=LABELS,
    value=LABELS[0],
    description="Label:"
)

notes_text = widgets.Text(
    value="",
    description="Notes:",
    placeholder="optional"
)

save_button = widgets.Button(description="Save label", button_style="success")
skip_button = widgets.Button(description="Skip", button_style="warning")
undo_button = widgets.Button(description="Undo last", button_style="danger")
export_button = widgets.Button(description="Export CSV/Parquet", button_style="info")

status_out = widgets.Output()
plot_out = widgets.Output()

def refresh_trial(change=None):
    with plot_out:
        clear_output(wait=True)
        session = session_dropdown.value
        t = trial_slider.value
        df_trial, s, e = get_trial_df(session, t)
        plot_trial_trajectory(df_trial, session=session, s=s, e=e)

    with status_out:
        clear_output(wait=True)
        print(f"Ready: session={session_dropdown.value}, trial={trial_slider.value}")

        # sanity check slice matches expected frames
        if len(df_trial) == 0:
            print("WARNING: empty slice (no rows matched this trial window)")
        else:
            print(f"Frames in slice: {df_trial[FRAME_COL].min()}–{df_trial[FRAME_COL].max()} (expected {s}–{e})")

def update_slider_range(change):
    session = change["new"]
    trial_slider.max = len(trials_by_session[session]) - 1
    trial_slider.value = 0
    refresh_trial()

def save_label(_):
    session = session_dropdown.value
    t = trial_slider.value
    (s, e) = map(int, trials_by_session[session][t])

    rec = dict(
        session_id=session,
        trial_idx=int(t),
        start_frame=int(s),
        end_frame=int(e),
        label=str(label_dropdown.value),
        notes=str(notes_text.value),
    )
    labels_records.append(rec)

    with status_out:
        clear_output(wait=True)
        print(f"Saved: {rec}")
        print(f"Total labeled: {len(labels_records)}")

    if trial_slider.value < trial_slider.max:
        trial_slider.value += 1
    else:
        with status_out:
            print("Reached last trial for this session.")

def skip_trial(_):
    if trial_slider.value < trial_slider.max:
        trial_slider.value += 1
    else:
        with status_out:
            clear_output(wait=True)
            print("Reached last trial for this session.")

def undo_last(_):
    if labels_records:
        removed = labels_records.pop()
        with status_out:
            clear_output(wait=True)
            print(f"Removed last label: {removed}")
            print(f"Total labeled: {len(labels_records)}")
    else:
        with status_out:
            clear_output(wait=True)
            print("Nothing to undo.")

def export_labels(_):
    if not labels_records:
        with status_out:
            clear_output(wait=True)
            print("No labels to export yet.")
        return

    df_labels = pd.DataFrame(labels_records)

    csv_path = "trajectory_trial_labels.csv"
    parquet_path = "trajectory_trial_labels.parquet"

    df_labels.to_csv(csv_path, index=False)
    df_labels.to_parquet(parquet_path, index=False)

    with status_out:
        clear_output(wait=True)
        print(f"Exported:\n- {csv_path}\n- {parquet_path}")
        display(df_labels.tail(10))

# Wire events
session_dropdown.observe(update_slider_range, names="value")
trial_slider.observe(refresh_trial, names="value")
save_button.on_click(save_label)
skip_button.on_click(skip_trial)
undo_button.on_click(undo_last)
export_button.on_click(export_labels)

# Display UI
controls = widgets.HBox([session_dropdown, trial_slider])
label_row = widgets.HBox([label_dropdown, notes_text])
buttons = widgets.HBox([save_button, skip_button, undo_button, export_button])

display(controls, label_row, buttons, plot_out, status_out)

# Initial render
refresh_trial()

Output()

Output()